## This is the code to train the model and acquire influence for Number of Samples Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [2]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [4]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [5]:
import random
from keras.optimizers import SGD

In [6]:
from sklearn.datasets import make_classification

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to test on different number of samples.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [7]:
train_pool = 16000
test_size = 500
train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
n_features=160
seed=42
sep = 3

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [8]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [9]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0     -13.782908  -6.132727  14.152943  -7.137928   2.798538 -29.164432   
1       6.337057  -5.322490  -7.760765  -6.599484  11.619708  -2.447431   
2      -4.162488   8.509963   6.350375   1.877896   8.781976  -5.668239   
3      -0.720892  -0.215843   3.809480  -4.197446   2.975266  -7.387154   
4       0.271039  -1.547205  -0.651251  -8.493556   8.239804  -5.990558   
...          ...        ...        ...        ...        ...        ...   
16495  -4.716227  -1.941982   4.511306  -3.178930  -8.376654  -7.079567   
16496   2.277247 -13.312607   8.519820   1.821284  -5.619789  -8.264994   
16497  -6.898158   0.233196  -8.868816  -3.021291   7.341948  13.122732   
16498  -1.610179 -15.301590   0.750310   4.510539   1.759556  -0.099331   
16499 -15.536142   3.698689   7.265016   1.931781   4.659502   8.521925   

       feature_7  feature_8  feature_9  feature_10  ...  feature_153  \
0      10.887783  -6.063796

In [10]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [11]:
print(df_train_pool.head())
print(df_test.head())

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0 -13.782908  -6.132727  14.152943  -7.137928   2.798538 -29.164432   
1   6.337057  -5.322490  -7.760765  -6.599484  11.619708  -2.447431   
2  -4.162488   8.509963   6.350375   1.877896   8.781976  -5.668239   
3  -0.720892  -0.215843   3.809480  -4.197446   2.975266  -7.387154   
4   0.271039  -1.547205  -0.651251  -8.493556   8.239804  -5.990558   

   feature_7  feature_8  feature_9  feature_10  ...  feature_153  feature_154  \
0  10.887783  -6.063796  19.086774   -3.520991  ...     5.819484   -10.721757   
1  -0.525553   7.558629   5.793142    2.989492  ...    -1.817321    -2.204365   
2 -15.248069   4.155376   1.674610    6.811081  ...     5.133064    14.611964   
3 -14.208869   0.459683  18.384366    9.957595  ...     4.951633     0.328660   
4 -14.237140  13.315085  10.821372   -3.751887  ...     2.459658     3.218489   

   feature_155  feature_156  feature_157  feature_158  feature_159  \
0    -2.265044  

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

In [12]:
features_to_test = 10
selected_features = [f'feature_{i+1}' for i in range(features_to_test)]

nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]
nested_train_dfs = [df[ selected_features + ['label', 'id'] ].copy()for df in nested_train_dfs]

df_test = df_test[ selected_features + ['label', 'id'] ].copy()

core_1000_ids = nested_train_dfs[0]['id'].tolist()

**The most important thing in this experiment is the following code**:  
Based on the train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000] defined above, we could map the number of training samples with the following code. By choosing the number in [], we could modify the train set size. Therefore, only changing the following code block is enough to produce the experiment result successfully.

In [13]:
train_df = nested_train_dfs[7]

In [14]:
train_df["clean_label"] = train_df["label"].copy()

# Randomly select 20% of the training samples
noise_ratio = 0.20
noise_seed = 42

rng = np.random.default_rng(noise_seed)

n_noisy = int(len(train_df) * noise_ratio)

noisy_positions = rng.choice(
    len(train_df),
    size=n_noisy,
    replace=False
)

# Indicator showing which samples were corrupted
train_df["is_noisy"] = 0
train_df.loc[train_df.index[noisy_positions], "is_noisy"] = 1

# Flip the binary labels: 0 -> 1 and 1 -> 0
train_df.loc[
    train_df.index[noisy_positions],
    "label"
] = 1 - train_df.loc[
    train_df.index[noisy_positions],
    "label"
]

# Save a clearer name for the labels used during training
train_df["noisy_label"] = train_df["label"]

print("Number of training samples:", len(train_df))
print("Number of flipped labels:", train_df["is_noisy"].sum())
print("Noise ratio:", train_df["is_noisy"].mean())

print(
    train_df[
        ["id", "clean_label", "noisy_label", "is_noisy"]
    ].head()
)

Number of training samples: 8000
Number of flipped labels: 1600
Noise ratio: 0.2
   id  clean_label  noisy_label  is_noisy
0   1            0            0         0
1   2            0            0         0
2   3            0            0         0
3   4            0            0         0
4   5            0            0         0


In [15]:
# Feature columns used by the model
selected_features = [
    col for col in train_df.columns
    if col.startswith("feature_")
]

# Preserve IDs separately
train_ids_original = train_df["id"].to_numpy()

# Scale IDs only for the influence pipeline
IDs = (
    train_ids_original
    .reshape(-1, 1)
    .astype(np.float32)
    / 1e10
)

# Model features
X_train_features = train_df[
    selected_features
].to_numpy(dtype=np.float32)

# Append the ID column, as required by your existing influence code
X_train = np.hstack([
    X_train_features,
    IDs
])

# Use the corrupted labels for training
y_train_1d = train_df[
    "noisy_label"
].to_numpy(dtype=np.int64)

y_train = to_categorical(
    y_train_1d,
    num_classes=2
)

print(X_train.shape)
print(y_train.shape)

(8000, 11)
(8000, 2)


In [16]:
# X_train = train_df.drop(columns=["label"])
# y_train = train_df["label"]
# IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
# IDs = IDs  / 1e10

# X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
# X_train = np.hstack((X_train, IDs))
# y_train = to_categorical(y_train.values,num_classes=2)

# print(X_train)

In [17]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)
print(y_test.shape)

(500, 11)
(500, 2)


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [18]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [19]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS
# import seaborn as sns
# import matplotlib.pyplot as plt

In [20]:
# D = pairwise_distances(X_all) 

In [21]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [22]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [23]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [24]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [25]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
initial_model = tf.keras.models.clone_model(model)
initial_model.set_weights(model.get_weights())
model_list.append(InfluenceModel(initial_model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  checkpoint_model = tf.keras.models.clone_model(model)
  checkpoint_model.set_weights(model.get_weights())
  model_list.append(InfluenceModel(checkpoint_model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.8587 - accuracy: 0.4848 - val_loss: 0.8211 - val_accuracy: 0.4640 - 709ms/epoch - 22ms/step
32/32 - 0s - loss: 0.7670 - accuracy: 0.4991 - val_loss: 0.7436 - val_accuracy: 0.5300 - 92ms/epoch - 3ms/step
32/32 - 0s - loss: 0.7260 - accuracy: 0.5244 - val_loss: 0.7063 - val_accuracy: 0.5580 - 71ms/epoch - 2ms/step
32/32 - 0s - loss: 0.7069 - accuracy: 0.5449 - val_loss: 0.6838 - val_accuracy: 0.5820 - 69ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6956 - accuracy: 0.5584 - val_loss: 0.6678 - val_accuracy: 0.6100 - 62ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6880 - accuracy: 0.5675 - val_loss: 0.6556 - val_accuracy: 0.6340 - 68ms/epoch - 2ms/step
32/32 - 0s - loss: 0.6824 - accuracy: 0.5765 - val_loss: 0.6456 - val_accuracy: 0.6480 - 93ms/epoch - 3ms/step
32/32 - 0s - loss: 0.6781 - accuracy: 0.5841 - val_loss: 0.6374 - val_accuracy: 0.6580 - 85ms/epoch - 3ms/step
32/32 - 0s - loss: 0.6745 - accuracy: 0.5884 - val_loss: 0.6305 - val_accuracy: 0.6660 - 68ms/epoch - 2ms/step

In [26]:
train_logits = model.predict(
    X_train,
    batch_size=256,
    verbose=0
)

# Calculate one loss value per sample using the corrupted labels
per_sample_loss_fn = CategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

training_losses = per_sample_loss_fn(
    y_train,
    train_logits
).numpy()

print(training_losses.shape)
print(pd.Series(training_losses).describe())

(8000,)
count    8000.000000
mean        0.614858
std         0.377039
min         0.030025
25%         0.335022
50%         0.500512
75%         0.825212
max         3.255857
dtype: float64


In [27]:
noise_loss_df = pd.DataFrame({
    "Train_ID": train_ids_original,
    "Clean_Label": train_df["clean_label"].to_numpy(),
    "Noisy_Label": train_df["noisy_label"].to_numpy(),
    "is_noisy": train_df["is_noisy"].to_numpy(),
    "Training_Loss": training_losses
})

print(noise_loss_df.head())
print(
    noise_loss_df.groupby("is_noisy")[
        "Training_Loss"
    ].describe()
)

   Train_ID  Clean_Label  Noisy_Label  is_noisy  Training_Loss
0         1            0            0         0       1.195575
1         2            0            0         0       0.334451
2         3            0            0         0       0.211052
3         4            0            0         0       0.268103
4         5            0            0         0       0.213011
           count      mean       std       min       25%       50%       75%  \
is_noisy                                                                       
0         6400.0  0.516793  0.288478  0.030025  0.309178  0.442105  0.658708   
1         1600.0  1.007118  0.430882  0.126030  0.672778  0.991035  1.283901   

               max  
is_noisy            
0         2.630126  
1         3.255857  


In [28]:
noise_loss_df.to_csv(
    "Noise_GroundTruth_and_TrainingLoss.csv",
    index=False
)

In [29]:
train_df.to_csv(
    "NoisyLabel_TrainingData.csv",
    index=False
)

test_df.to_csv(
    "Clean_TestData.csv",
    index=False
)

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [30]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [31]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0            1  0.130482
1            2  0.462659
2            3  0.443563
3            4  0.447989
4            5  0.384325
...        ...       ...
7995      7996  0.269913
7996      7997  0.332257
7997      7998  0.135116
7998      7999  0.278339
7999      8000  0.272090

[8000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [32]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0            1  0.200748
1            2  0.150704
2            3  0.108237
3            4  0.122400
4            5  0.081591
...        ...       ...
7995      7996  0.105113
7996      7997  0.208628
7997      7998  0.116131
7998      7999  0.156522
7999      8000 -0.129287

[8000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [33]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [34]:
TracIn_sorted.to_csv("NoisyLabel_TracIn_Scores.csv",index = False)
df_sorted.to_csv("NoisyLabel_FOIF_Scores.csv",index = False)